# Lab Day22 — tracking trên năm video
Detector và Re-ID giữ nguyên theo đề. Chỉ video_1 có số liệu chấm.
Bản preflight chỉ kiểm tra môi trường, không tạo kết quả bài nộp.

In [ ]:
import json, os, subprocess, sys, shutil
from pathlib import Path
WORK = Path('/kaggle/working/day22')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SOURCES = {'scripts/archive_parts.py': '"""Chia và ghép ZIP theo byte, xác minh checksum từng phần và toàn bộ."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom submission_checks import validate_zip_members\n\n\ndef split_archive(archive: Path, destination: Path, part_bytes: int) -> dict:\n    """Ghi các phần mới và manifest, giữ nguyên tệp nguồn.\n\n    Args:\n        archive: ZIP gốc.\n        destination: Thư mục staging đã tạo.\n        part_bytes: Số byte tối đa của một phần.\n\n    Returns:\n        Manifest chứa kích thước và SHA-256.\n\n    Raises:\n        ValueError: Khi kích thước phần không dương.\n        FileExistsError: Khi đích đã có tệp cùng tên.\n    """\n    if part_bytes <= 0:\n        raise ValueError(\'Kích thước phần phải dương.\')\n    parts = []\n    digest = hashlib.sha256()\n    with archive.open(\'rb\') as source:\n        while data := source.read(part_bytes):\n            name = f\'data_lab21.zip.part{len(parts):04d}.bin\'\n            with (destination / name).open(\'xb\') as target:\n                target.write(data)\n            digest.update(data)\n            parts.append({\'name\': name, \'bytes\': len(data), \'sha256\': hashlib.sha256(data).hexdigest()})\n    manifest = {\'bytes\': archive.stat().st_size, \'sha256\': digest.hexdigest(), \'parts\': parts}\n    with (destination / \'data_parts.json\').open(\'x\') as target:\n        json.dump(manifest, target, indent=2)\n    return manifest\n\n\ndef assemble_archive(manifest_path: Path, destination: Path) -> None:\n    """Ghép vào tệp mới và kiểm tra từng phần cùng checksum ZIP gốc.\n\n    Args:\n        manifest_path: JSON đi cùng các phần dữ liệu.\n        destination: ZIP mới, chưa tồn tại.\n\n    Raises:\n        ValueError: Khi đường dẫn, kích thước hoặc checksum không hợp lệ.\n        FileExistsError: Khi ZIP đích đã tồn tại.\n    """\n    manifest = json.loads(manifest_path.read_text())\n    validate_zip_members([part[\'name\'] for part in manifest[\'parts\']])\n    digest = hashlib.sha256()\n    total = 0\n    with destination.open(\'xb\') as target:\n        for part in manifest[\'parts\']:\n            data = (manifest_path.parent / part[\'name\']).read_bytes()\n            if len(data) != part[\'bytes\'] or hashlib.sha256(data).hexdigest() != part[\'sha256\']:\n                raise ValueError(f"Phần dữ liệu lỗi checksum: {part[\'name\']}")\n            target.write(data)\n            digest.update(data)\n            total += len(data)\n    if total != manifest[\'bytes\'] or digest.hexdigest() != manifest[\'sha256\']:\n        raise ValueError(\'ZIP ghép không khớp checksum gốc.\')\n', 'scripts/build_kaggle_notebook.py': '"""Đóng gói code hiện tại thành notebook chạy trong phiên Kaggle riêng."""\n\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef main() -> None:\n    """Tạo notebook và metadata riêng tư, không upload hoặc chạy từ xa."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--owner", required=True)\n    parser.add_argument("--dataset", default="")\n    parser.add_argument("--preflight", action="store_true")\n    parser.add_argument("--selections", type=Path, help="JSON cấu hình đã chọn sau khi xem video thử")\n    args = parser.parse_args()\n    root = Path(__file__).resolve().parents[1]\n    target = root / "kaggle" / ("preflight" if args.preflight else "lab")\n    target.mkdir(parents=True, exist_ok=True)\n    cells = []\n    cells.append({"cell_type": "markdown", "metadata": {}, "source": [\n        "# Lab Day22 — tracking trên năm video\\n",\n        "Detector và Re-ID giữ nguyên theo đề. Chỉ video_1 có số liệu chấm.\\n",\n        "Bản preflight chỉ kiểm tra môi trường, không tạo kết quả bài nộp."\n    ]})\n    sources = {}\n    for pattern in ("scripts/*.py", "tests/*.py"):\n        for path in sorted(root.glob(pattern)):\n            sources[str(path.relative_to(root))] = path.read_text()\n    sources["pytest.ini"] = (root / "pytest.ini").read_text()\n    sources["on_tap_metrics.ipynb"] = (root / "on_tap_metrics.ipynb").read_text()\n    sources["requirements-kaggle.txt"] = (root / "requirements-kaggle.txt").read_text()\n    setup = \'\'\'import json, os, subprocess, sys, shutil\nfrom pathlib import Path\nWORK = Path(\'/kaggle/working/day22\')\nWORK.mkdir(parents=True, exist_ok=True)\nos.chdir(WORK)\n\'\'\'\n    setup += "SOURCES = " + repr(sources) + "\\n"\n    setup += \'\'\'for name, content in SOURCES.items():\n    destination = WORK / name\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(content)\nfor weight in [\'yolo26n.pt\', \'osnet_x0_25_msmt17.pt\']:\n    cached = list(Path(\'/kaggle/input\').rglob(weight))\n    if len(cached) == 1 and not (WORK / weight).exists():\n        shutil.copy2(cached[0], WORK / weight)\n        print(f\'Dùng lại trọng số đã kiểm thử: {weight}\')\n# Kaggle hiện chạy Python 3.13: giữ thư viện GPU có sẵn của phiên.\n# BoxMOT cũ pin NumPy 1.23.1 không có wheel tương thích; cài riêng\n# không kéo dependency, rồi kiểm tra đủ năm tracker với NumPy của Kaggle.\nPYTHON = sys.executable\nsubprocess.run([PYTHON, \'-m\', \'pip\', \'install\', \'-r\', \'requirements-kaggle.txt\'], check=True)\nsubprocess.run([PYTHON, \'-m\', \'pip\', \'install\', \'--no-deps\', \'boxmot==10.0.42\'], check=True)\nsubprocess.run([PYTHON, \'-m\', \'pytest\', \'-q\'], check=True)\nsubprocess.run([PYTHON, \'-m\', \'pip\', \'freeze\'], check=True, stdout=(WORK / \'versions.txt\').open(\'w\'))\n\'\'\'\n    cells.append({"cell_type": "code", "metadata": {}, "source": setup.splitlines(True), "outputs": [], "execution_count": None})\n    if args.preflight:\n        smoke = \'\'\'import textwrap\nsmoke_code = r"""\nimport json\nfrom pathlib import Path\nimport cv2, numpy as np, torch\nfrom ultralytics import YOLO\nfrom boxmot.tracker_zoo import create_tracker, get_tracker_config\nassert torch.cuda.is_available(), \'Chưa có GPU\'\nmodel = YOLO(\'yolo26n.pt\').to(\'cuda:0\')\nmodel.predict(np.zeros((640, 640, 3), dtype=np.uint8), imgsz=640, classes=[0], device=\'cuda:0\', verbose=False)\nresults = {}\nfor name in [\'bytetrack\', \'ocsort\', \'botsort\', \'strongsort\', \'deepocsort\']:\n    tracker = create_tracker(name, get_tracker_config(name), Path(\'osnet_x0_25_msmt17.pt\'), torch.device(\'cuda:0\'), False, False)\n    frame = np.full((640, 640, 3), 127, dtype=np.uint8)\n    for i in range(5):\n        tracks = tracker.update(np.array([[100+i, 100, 200+i, 300, 0.95, 0.0]]), frame)\n    assert tracks.ndim == 2 and tracks.shape[1] >= 6, (name, tracks.shape)\n    results[name] = list(tracks.shape)\nsample = Path(\'/tmp/day22_smoke_frames\')\nsample.mkdir(exist_ok=False)\nfor i in range(5):\n    cv2.imwrite(str(sample / f\'{i+1:06d}.jpg\'), np.zeros((640, 640, 3), dtype=np.uint8))\nimport subprocess, sys\nfor name in [\'bytetrack\', \'botsort\']:\n    command = [sys.executable, \'scripts/run_tracking.py\', \'--source\', str(sample),\n        \'--seq-name\', \'kiem_thu\', \'--tracker\', name, \'--device\', \'cuda:0\',\n        \'--out\', str(Path(\'/tmp\') / (\'day22_smoke_\' + name)), \'--save-video\']\n    result = subprocess.run(command, capture_output=True, text=True, check=True)\n    assert \'-> 5 frame \' in result.stdout, result.stdout\n    print(result.stdout)\nPath(\'preflight.json\').write_text(json.dumps({\'gpu\': torch.cuda.get_device_name(), \'trackers\': results, \'status\': \'passed\'}, indent=2))\nprint(results)\n"""\nsubprocess.run([PYTHON, \'-c\', smoke_code], check=True)\ntrackeval = Path(\'/tmp/day22_trackeval_smoke\')\nsubprocess.run([\'git\', \'clone\', \'https://github.com/JonathonLuiten/TrackEval.git\', str(trackeval)], check=True)\nsubprocess.run([PYTHON, \'-m\', \'pip\', \'install\', \'--no-deps\', \'-e\', str(trackeval)], check=True)\npractice = Path(\'/tmp/day22_eval_smoke/video_1\')\n(practice / \'gt\').mkdir(parents=True)\nrows = \'\\\\n\'.join(f\'{i},1,10,10,30,60,1,1,1\' for i in range(1, 6)) + \'\\\\n\'\n(practice / \'gt/gt.txt\').write_text(rows)\n(practice / \'seqinfo.ini\').write_text(\'[Sequence]\\\\nname=video_1\\\\nimDir=img1\\\\nframeRate=20\\\\nseqLength=5\\\\nimWidth=640\\\\nimHeight=640\\\\nimExt=.jpg\\\\n\')\n(practice / \'eval_config.json\').write_text(json.dumps({\'benchmark\': \'LAB\', \'split\': \'train\'}))\nsubmission = practice.parent / \'video_1.txt\'\nsubmission.write_text(\'\\\\n\'.join(f\'{i},1,10,10,30,60,1,-1,-1,-1\' for i in range(1, 6)) + \'\\\\n\')\nwith (WORK / \'preflight_trackeval.log\').open(\'w\') as log:\n    subprocess.run([PYTHON, \'scripts/evaluate_practice.py\', \'--trackeval-root\', str(trackeval),\n        \'--lab-data-root\', str(practice.parent), \'--submission\', str(submission), \'--run-name\', \'smoke\'],\n        stdout=log, stderr=subprocess.STDOUT, check=True)\n(WORK / \'trackeval_preflight_passed.txt\').write_text(\'Kiểm tra tương thích bằng dữ liệu tổng hợp; không phải metric bài lab.\\\\n\')\n\'\'\'\n        cells.append({"cell_type": "code", "metadata": {}, "source": smoke.splitlines(True), "outputs": [], "execution_count": None})\n    else:\n        code = ""\n        if args.selections:\n            choices = json.loads(args.selections.read_text())\n            code += "(WORK / \'selections.json\').write_text(" + repr(json.dumps(choices)) + ")\\n"\n            code += "os.environ[\'LAB_MODE\'] = \'final\'\\n"\n        code += "subprocess.run([PYTHON, \'scripts/kaggle_lab_pipeline.py\'], check=True)\\n"\n        cells.append({"cell_type": "code", "metadata": {}, "source": code.splitlines(True), "outputs": [], "execution_count": None})\n    notebook = {"cells": cells, "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}}, "nbformat": 4, "nbformat_minor": 4}\n    (target / "day22.ipynb").write_text(json.dumps(notebook, ensure_ascii=False, indent=2))\n    slug = "track4-day22-tracking" + ("-preflight" if args.preflight else "")\n    metadata = {"id": args.owner + "/" + slug, "title": "Track4 Day22 Tracking" + (" Preflight" if args.preflight else ""),\n        "code_file": "day22.ipynb", "language": "python", "kernel_type": "notebook",\n        "is_private": True, "enable_gpu": True, "enable_internet": True,\n        "dataset_sources": [args.dataset] if args.dataset else [], "competition_sources": [],\n        "kernel_sources": [] if args.preflight else [args.owner + \'/track4-day22-tracking-preflight\']}\n    (target / "kernel-metadata.json").write_text(json.dumps(metadata, indent=2))\n    print(f"Đã tạo notebook tại {target}")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/check_data.py': '#!/usr/bin/env python\n"""Kiểm tra gói lab_data giảng viên phát đã đủ ảnh (và nhãn video luyện) chưa.\n\nVí dụ:\n    python scripts/check_data.py --lab-data-root "$LAB_DATA"\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nVIDEOS = ["video_1", "video_2", "video_3", "video_4", "video_5"]\nPRACTICE_VIDEO = "video_1"\n\nCONTEXT = {\n    "video_1": "Quảng trường ngoài trời, camera TĨNH, ban ngày, mật độ trung bình — video luyện, có nhãn",\n    "video_2": "Phố, camera TĨNH trên cao, BAN ĐÊM, mật độ RẤT đông",\n    "video_3": "Camera DI CHUYỂN, độ phân giải thấp, khung hình chậm",\n    "video_4": "TRONG NHÀ, camera di chuyển tiến về phía trước, phản chiếu kính",\n    "video_5": "Quay từ XE BUS tại giao lộ đông, rung lắc mạnh",\n}\n\n\ndef check(lab_data_root: Path) -> bool:\n    """In trạng thái từng video và báo gói lab có dùng được không.\n\n    Args:\n        lab_data_root: Thư mục lab_data giảng viên phát.\n\n    Returns:\n        True nếu cả năm video có ảnh và video_1 có file nhãn.\n    """\n    all_ok = True\n    for name in VIDEOS:\n        img_dir = lab_data_root / name / "img1"\n        n_imgs = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0\n        gt_file = lab_data_root / name / "gt" / "gt.txt"\n        has_gt = gt_file.exists()\n        if name == PRACTICE_VIDEO:\n            ok = n_imgs > 0 and has_gt\n            gt_note = "có nhãn" if has_gt else "THIẾU nhãn video luyện"\n        else:\n            ok = n_imgs > 0\n            gt_note = "không có nhãn (đúng)" if not has_gt else "có nhãn — không dùng để tự chấm"\n        all_ok &= ok\n        status = "OK   " if ok else "THIẾU"\n        print(f"[{status}] {name:8s}  {n_imgs:4d} ảnh  {gt_note}  — {CONTEXT[name]}")\n    preview_dir = lab_data_root / "preview"\n    n_preview = len(list(preview_dir.glob("*.mp4"))) if preview_dir.exists() else 0\n    print(f"\\nPreview .mp4: {n_preview} file trong {preview_dir}")\n    if all_ok:\n        print("Gói dữ liệu đủ để chạy scripts/run_tracking.py.")\n    else:\n        print("Thiếu ảnh hoặc thiếu nhãn video_1. Nhờ giảng viên phát lại gói lab_data.")\n    return all_ok\n\n\ndef main() -> None:\n    """Đọc ``--lab-data-root`` và in kết quả kiểm tra gói dữ liệu."""\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--lab-data-root", required=True, type=Path, help="Thư mục lab_data giảng viên phát")\n    args = parser.parse_args()\n    check(args.lab_data_root)\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/evaluate_practice.py': '#!/usr/bin/env python\n"""Chấm thử MỘT video luyện (video_1) để đọc HOTA / MOTA / IDF1.\n\nScript chỉ nhận video_1. Bốn video còn lại không có nhãn trong gói học viên —\nđánh giá chúng bằng mắt, ghi vào báo cáo.\n\nYêu cầu trước:\n    git clone https://github.com/JonathonLuiten/TrackEval.git\n    pip install -e TrackEval/\n\nVí dụ:\n    python scripts/evaluate_practice.py \\\\\n        --trackeval-root ~/TrackEval \\\\\n        --lab-data-root "$LAB_DATA" \\\\\n        --submission "$LAB_DATA/../runs/nop_bai/video_1.txt" \\\\\n        --run-name nhom01_video1\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nPRACTICE_VIDEO = "video_1"\n\n\ndef _patch_numpy_aliases() -> None:\n    """TrackEval còn gọi np.float / np.int (đã bỏ từ NumPy 1.24)."""\n    import numpy as np\n\n    if not hasattr(np, "float"):\n        np.float = float  # type: ignore[attr-defined]\n    if not hasattr(np, "int"):\n        np.int = int  # type: ignore[attr-defined]\n\n\ndef _load_eval_config(lab_data_root: Path) -> dict:\n    """Đọc cấu hình chấm đi kèm nhãn video luyện.\n\n    Args:\n        lab_data_root: Thư mục lab_data giảng viên phát.\n\n    Returns:\n        Dict có khóa ``benchmark`` và có thể có ``split``.\n\n    Raises:\n        json.JSONDecodeError: Khi file cấu hình hiện có không phải JSON hợp lệ.\n    """\n    config_path = lab_data_root / PRACTICE_VIDEO / "eval_config.json"\n    if not config_path.exists():\n        print("Gói dữ liệu thiếu eval_config.json; dùng benchmark LAB, split train. "\n              "Giữ nguyên gt.txt và seqinfo.ini của giảng viên.")\n        return {"benchmark": "LAB", "split": "train"}\n    return json.loads(config_path.read_text())\n\n\ndef stage(trackeval_root: Path, lab_data_root: Path, submission: Path, run_name: str, benchmark: str, split: str = "train") -> None:\n    """Copy nhãn video luyện và file nộp vào cây thư mục TrackEval.\n\n    Args:\n        trackeval_root: Thư mục gốc bản clone TrackEval.\n        lab_data_root: Thư mục lab_data giảng viên phát.\n        submission: File ``video_1.txt`` do ``run_tracking.py`` sinh ra.\n        run_name: Tên lần chấm, dùng làm thư mục tracker.\n        benchmark: Tên benchmark TrackEval, lấy từ ``eval_config.json``.\n        split: Nhánh dữ liệu khai báo trong cấu hình chấm.\n\n    Raises:\n        FileNotFoundError: Khi thiếu nhãn, ``seqinfo.ini``, hoặc file nộp.\n    """\n    src = lab_data_root / PRACTICE_VIDEO\n    gt_file = src / "gt" / "gt.txt"\n    seqinfo = src / "seqinfo.ini"\n    if not gt_file.exists() or not seqinfo.exists():\n        raise FileNotFoundError(f"video luyện thiếu gt hoặc seqinfo trong {src}")\n    if not submission.exists():\n        raise FileNotFoundError(f"Không thấy file nộp {submission}")\n\n    split_dir = f"{benchmark}-{split}"\n    gt_dst = trackeval_root / "data" / "gt" / "mot_challenge" / split_dir / PRACTICE_VIDEO\n    (gt_dst / "gt").mkdir(parents=True, exist_ok=True)\n    shutil.copy(gt_file, gt_dst / "gt" / "gt.txt")\n    shutil.copy(seqinfo, gt_dst / "seqinfo.ini")\n\n    sub_dst = trackeval_root / "data" / "trackers" / "mot_challenge" / split_dir / run_name / "data"\n    sub_dst.mkdir(parents=True, exist_ok=True)\n    shutil.copy(submission, sub_dst / f"{PRACTICE_VIDEO}.txt")\n\n\ndef run_trackeval(trackeval_root: Path, run_name: str, benchmark: str, split: str) -> None:\n    """Gọi script chấm của TrackEval, chỉ một video luyện.\n\n    Args:\n        trackeval_root: Thư mục gốc bản clone TrackEval.\n        run_name: Tên lần chấm đã stage.\n        benchmark: Tên benchmark TrackEval.\n        split: Nhánh dữ liệu, thường là ``train``.\n\n    Raises:\n        subprocess.CalledProcessError: Khi TrackEval thoát với mã khác 0.\n    """\n    cmd = [\n        sys.executable,\n        "-c",\n        "import sys, runpy, numpy as np; "\n        "np.float = float; np.int = int; "\n        "entry = sys.argv.pop(1); sys.argv[0] = entry; "\n        "runpy.run_path(entry, run_name=\'__main__\')",\n        str(trackeval_root / "scripts" / "run_mot_challenge.py"),\n        "--GT_FOLDER", str(trackeval_root / "data" / "gt" / "mot_challenge"),\n        "--TRACKERS_FOLDER", str(trackeval_root / "data" / "trackers" / "mot_challenge"),\n        "--BENCHMARK", benchmark,\n        "--SPLIT_TO_EVAL", split,\n        "--SEQ_INFO", PRACTICE_VIDEO,\n        "--TRACKERS_TO_EVAL", run_name,\n        "--METRICS", "HOTA", "CLEAR", "Identity",\n        "--USE_PARALLEL", "False",\n    ]\n    print("Đang chấm video luyện:\\n  " + " ".join(cmd) + "\\n")\n    subprocess.run(cmd, check=True)\n\n\ndef main() -> None:\n    """Chấm file ``video_1.txt`` và in HOTA, MOTA, IDF1.\n\n    Raises:\n        SystemExit: Khi file nộp không phải ``video_1.txt``.\n    """\n    _patch_numpy_aliases()\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--trackeval-root", required=True, type=Path)\n    parser.add_argument("--lab-data-root", required=True, type=Path)\n    parser.add_argument("--submission", required=True, type=Path, help="File video_1.txt do run_tracking.py sinh ra")\n    parser.add_argument("--run-name", required=True, help="Tên lần chấm, ví dụ nhom01_video1")\n    args = parser.parse_args()\n\n    if args.submission.name != f"{PRACTICE_VIDEO}.txt":\n        raise SystemExit(\n            f"Script này chỉ chấm {PRACTICE_VIDEO}.txt. "\n            "video_2 đến video_5 không có nhãn — hãy đánh giá bằng mắt."\n        )\n\n    config = _load_eval_config(args.lab_data_root)\n    benchmark = config["benchmark"]\n    split = config.get("split", "train")\n    stage(args.trackeval_root, args.lab_data_root, args.submission, args.run_name, benchmark, split)\n    run_trackeval(args.trackeval_root, args.run_name, benchmark, split)\n    print(\n        "\\nĐọc bảng phía trên: HOTA cân bằng phát hiện và giữ danh tính; "\n        "MOTA phạt số lần đổi ID; IDF1 nhạy với track dài bị gán sai ID. "\n        "Dùng các số này cho video_1 trong báo cáo. Không có bảng số cho video khác."\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/inspect_tracks.py': '"""Tạo bảng ảnh có ID từ kết quả MOT để so sánh trực quan các cấu hình."""\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef main() -> None:\n    """Vẽ các frame được chỉ định, giữ nguyên ảnh và kết quả nguồn.\n\n    Raises:\n        FileExistsError: Khi ảnh đích đã tồn tại.\n        ValueError: Khi chỉ số frame vượt số ảnh của video.\n    """\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--lab-data-root\', required=True, type=Path)\n    parser.add_argument(\'--video\', required=True)\n    parser.add_argument(\'--result\', required=True, action=\'append\', type=Path)\n    parser.add_argument(\'--frames\', default=\'1,50,100,150\')\n    parser.add_argument(\'--output\', required=True, type=Path)\n    args = parser.parse_args()\n    if args.output.exists():\n        raise FileExistsError(args.output)\n    frames = [int(value) for value in args.frames.split(\',\')]\n    images = sorted((args.lab_data_root / args.video / \'img1\').glob(\'*.jpg\'))\n    if any(frame < 1 or frame > len(images) for frame in frames):\n        raise ValueError(\'Frame vượt phạm vi video.\')\n    font = ImageFont.truetype(\'/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\', 14)\n    sheet = Image.new(\'RGB\', (640 * len(frames), 400 * len(args.result)), \'white\')\n    for row, result in enumerate(args.result):\n        records = {}\n        for line in result.read_text().splitlines():\n            fields = line.split(\',\')\n            frame = int(fields[0])\n            if frame in frames:\n                records.setdefault(frame, []).append([float(value) for value in fields[:7]])\n        for column, frame in enumerate(frames):\n            original = Image.open(images[frame - 1]).convert(\'RGB\')\n            scale = min(640 / original.width, 360 / original.height)\n            tile = original.resize((round(original.width * scale), round(original.height * scale)))\n            draw = ImageDraw.Draw(tile)\n            for values in records.get(frame, []):\n                _, track_id, x, y, width, height, confidence = values\n                rng = np.random.default_rng(int(track_id) * 9973 + 17)\n                color = tuple(int(value) for value in rng.integers(64, 255, size=3)[::-1])\n                box = (x * scale, y * scale, (x + width) * scale, (y + height) * scale)\n                draw.rectangle(box, outline=color, width=2)\n                if height * scale >= 15:\n                    draw.text((max(0, x * scale), max(0, y * scale - 15)), str(int(track_id)),\n                              font=font, fill=color, stroke_width=1, stroke_fill=\'black\')\n            x0, y0 = column * 640, row * 400\n            sheet.paste(tile, (x0, y0 + 35))\n            ImageDraw.Draw(sheet).text((x0 + 5, y0 + 5),\n                f\'{result.parent.name} | frame {frame}\', font=font, fill=\'black\')\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(args.output)\n    print(f\'Đã lưu ảnh so sánh: {args.output}\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/kaggle_lab_pipeline.py': '"""Chạy thử hoặc chạy bản nộp trên Kaggle từ đúng gói dữ liệu lab."""\n\nimport json\nimport configparser\nimport hashlib\nimport os\nimport subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nfrom submission_checks import parse_metric_summary, validate_mot_rows, validate_zip_members\nfrom archive_parts import assemble_archive\n\n\ndef evaluate(trackeval: Path, lab: Path, submission: Path, run_name: str, output: Path) -> dict:\n    """Chấm một file video_1 và xác minh có summary hợp lệ.\n\n    Args:\n        trackeval: Thư mục cài TrackEval.\n        lab: Gốc dữ liệu lab.\n        submission: File video_1.txt đã chạy đủ frame.\n        run_name: Tên riêng cho lượt chấm.\n        output: Thư mục lưu log và summary.\n\n    Returns:\n        Dict metric HOTA, MOTA và IDF1 theo phần trăm.\n\n    Raises:\n        RuntimeError: Khi TrackEval không tạo summary duy nhất.\n    """\n    output.mkdir(parents=True, exist_ok=True)\n    with (output / \'metrics_video_1.log\').open(\'w\') as log:\n        subprocess.run([sys.executable, \'scripts/evaluate_practice.py\', \'--trackeval-root\', str(trackeval),\n            \'--lab-data-root\', str(lab), \'--submission\', str(submission), \'--run-name\', run_name],\n            stdout=log, stderr=subprocess.STDOUT, check=True)\n    summaries = list((trackeval / \'data/trackers/mot_challenge\').glob(f\'*/{run_name}/pedestrian_summary.txt\'))\n    if len(summaries) != 1:\n        raise RuntimeError(\'TrackEval không tạo được summary duy nhất.\')\n    content = summaries[0].read_text()\n    metrics = parse_metric_summary(content)\n    (output / \'metrics_video_1_summary.txt\').write_text(content)\n    (output / \'metrics_video_1.json\').write_text(json.dumps(metrics, indent=2))\n    return metrics\n\n\ndef main() -> None:\n    """Thử cấu hình, ghi log và kiểm tra bản nộp đủ frame trước khi đóng gói."""\n    work = Path.cwd()\n    candidates = [path for path in Path(\'/kaggle/input\').rglob(\'video_1/img1\')\n                  if \'__MACOSX\' not in path.parts]\n    if not candidates:\n        archives = list(Path(\'/kaggle/input\').rglob(\'data_lab21.zip\'))\n        if not archives:\n            manifests = list(Path(\'/kaggle/input\').rglob(\'data_parts.json\'))\n            if len(manifests) == 1:\n                joined = Path(\'/tmp/day22_data_lab21.zip\')\n                assemble_archive(manifests[0], joined)\n                archives = [joined]\n                print(\'Đã ghép dữ liệu, SHA-256 khớp ZIP gốc.\', flush=True)\n        if len(archives) != 1:\n            raise RuntimeError(\'Cần gắn đúng một gói data_lab21.zip hoặc cây ảnh lab.\')\n        destination = Path(\'/tmp/day22_lab_data_extracted\')\n        destination.mkdir(exist_ok=False)\n        with zipfile.ZipFile(archives[0]) as archive:\n            validate_zip_members(archive.namelist())\n            archive.extractall(destination)\n        candidates = [path for path in destination.rglob(\'video_1/img1\')\n                      if \'__MACOSX\' not in path.parts]\n    if len(candidates) != 1:\n        raise RuntimeError(\'Không xác định được duy nhất thư mục ảnh của lab.\')\n    lab = candidates[0].parent.parent\n    os.environ[\'LAB_DATA\'] = str(lab)\n    videos = [f\'video_{i}\' for i in range(1, 6)]\n    counts = {}\n    for video in videos:\n        frames = sorted((lab / video / \'img1\').glob(\'*.jpg\'))\n        if not frames:\n            raise RuntimeError(f\'{video}: thiếu ảnh\')\n        if video != \'video_1\' and (lab / video / \'gt\' / \'gt.txt\').exists():\n            raise RuntimeError(\'Gói học viên không được có nhãn video_2–video_5.\')\n        counts[video] = len(frames)\n    for name in [\'gt/gt.txt\', \'seqinfo.ini\']:\n        if not (lab / \'video_1\' / name).is_file():\n            raise RuntimeError(f\'Thiếu video_1/{name}\')\n    (work / \'data_inventory.json\').write_text(json.dumps(counts, indent=2))\n    subprocess.run([sys.executable, \'scripts/check_data.py\', \'--lab-data-root\', str(lab)], check=True)\n    subprocess.run([sys.executable, \'-m\', \'nbconvert\', \'--to\', \'notebook\', \'--execute\',\n        \'--ExecutePreprocessor.timeout=300\', \'--output\', \'on_tap_metrics_da_chay.ipynb\',\n        \'on_tap_metrics.ipynb\'], check=True)\n    mode = os.environ.get(\'LAB_MODE\', \'trials\')\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Bài chạy Kaggle chưa có GPU.\')\n    (work / \'runtime_info.json\').write_text(json.dumps({\n        \'python\': sys.version, \'gpu\': torch.cuda.get_device_name(0), \'mode\': mode}, indent=2))\n    sequence = configparser.ConfigParser()\n    sequence.read(lab / \'video_1/seqinfo.ini\')\n    fps_video1 = sequence.getint(\'Sequence\', \'frameRate\')\n    trackeval = Path(\'/tmp/day22_TrackEval\')\n    subprocess.run([\'git\', \'clone\', \'https://github.com/JonathonLuiten/TrackEval.git\', str(trackeval)], check=True)\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--no-deps\', \'-e\', str(trackeval)], check=True)\n    with (work / \'trackeval_commit.txt\').open(\'w\') as record:\n        subprocess.run([\'git\', \'-C\', str(trackeval), \'rev-parse\', \'HEAD\'], stdout=record, check=True)\n    results = []\n    if mode == \'trials\':\n        configs = [\n            (\'bytetrack\', .3, .5), (\'botsort\', .3, .5),\n            (\'botsort\', .15, .5), (\'botsort\', .5, .5),\n            (\'botsort\', .3, .4), (\'botsort\', .3, .7),\n            (\'bytetrack\', .15, .5), (\'bytetrack\', .5, .5),\n            (\'bytetrack\', .3, .4), (\'bytetrack\', .3, .7),\n        ]\n        for video in videos:\n            for tracker, conf, iou in configs:\n                label = f\'{tracker}_conf{conf}_iou{iou}\'\n                out = work / \'runs\' / \'thu_nghiem\' / label\n                out.mkdir(parents=True, exist_ok=True)\n                limit = counts[video] if video == \'video_1\' and conf == .3 and iou == .5 else min(150, counts[video])\n                command = [sys.executable, \'scripts/run_tracking.py\', \'--source\', str(lab / video / \'img1\'),\n                    \'--seq-name\', video, \'--tracker\', tracker, \'--conf\', str(conf), \'--iou\', str(iou),\n                    \'--out\', str(out), \'--save-video\', \'--device\', \'cuda:0\', \'--max-frames\', str(limit),\n                    \'--fps\', str(fps_video1 if video == \'video_1\' else 20)]\n                print(f\'Đang thử {video}: {label}\', flush=True)\n                with (out / f\'{video}.log\').open(\'w\') as log:\n                    subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)\n                validate_mot_rows((out / f\'{video}.txt\').read_text(), limit)\n                metrics = None\n                if video == \'video_1\' and limit == counts[video]:\n                    metrics = evaluate(trackeval, lab, out / \'video_1.txt\', label, out)\n                results.append({\'video\': video, \'tracker\': tracker, \'conf\': conf, \'iou\': iou,\n                    \'processed_frames\': limit, \'output\': str(out.relative_to(work)), \'metrics\': metrics})\n                (work / \'experiments.json\').write_text(json.dumps(results, indent=2))\n        (work / \'experiments.json\').write_text(json.dumps(results, indent=2))\n        print(\'Đã chạy thử. Cần xem video, chọn cấu hình rồi chạy chế độ final.\')\n    elif mode == \'final\':\n        selections = json.loads((work / \'selections.json\').read_text())\n        if set(selections) != set(videos):\n            raise RuntimeError(\'Cấu hình cuối phải chứa đủ năm video.\')\n        out = work / \'runs\' / \'nop_bai\'\n        out.mkdir(parents=True, exist_ok=True)\n        for video in videos:\n            cfg = selections[video]\n            command = [sys.executable, \'scripts/run_tracking.py\', \'--source\', str(lab / video / \'img1\'),\n                \'--seq-name\', video, \'--tracker\', cfg[\'tracker\'], \'--conf\', str(cfg[\'conf\']), \'--iou\', str(cfg[\'iou\']),\n                \'--out\', str(out), \'--save-video\', \'--device\', \'cuda:0\',\n                \'--fps\', str(fps_video1 if video == \'video_1\' else 20)]\n            print(f\'Đang chạy đủ frame {video}: {cfg}\', flush=True)\n            with (out / f\'{video}.log\').open(\'w\') as log:\n                subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)\n            validate_mot_rows((out / f\'{video}.txt\').read_text(), counts[video])\n            log_text = (out / f\'{video}.log\').read_text()\n            if f\'-> {counts[video]} frame \' not in log_text:\n                raise RuntimeError(f\'{video}: chưa chứng minh đã xử lý đủ ảnh.\')\n        evaluate(trackeval, lab, out / \'video_1.txt\', \'day22_final\', work)\n        print(\'Đã chạy đủ frame và chấm video_1. Cần hoàn thiện báo cáo từ video thực tế.\')\n    else:\n        raise RuntimeError(\'LAB_MODE phải là trials hoặc final.\')\n    weight_hashes = {}\n    for name in [\'yolo26n.pt\', \'osnet_x0_25_msmt17.pt\']:\n        with (work / name).open(\'rb\') as weight:\n            weight_hashes[name] = hashlib.file_digest(weight, \'sha256\').hexdigest()\n    (work / \'weights_sha256.json\').write_text(json.dumps(weight_hashes, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/kaggle_live_logs.py': '"""Đọc log phiên Kaggle trong thời gian hữu hạn và giữ bản sao mới."""\n\nimport argparse\nimport json\nimport signal\nfrom pathlib import Path\n\nfrom kaggle.api.kaggle_api_extended import KaggleApi\nfrom kagglesdk.kernels.types.kernels_api_service import ApiGetKernelSessionLogsStreamRequest\n\n\ndef main() -> None:\n    """Lưu log trực tiếp, không ghi đè tệp hoặc thay đổi phiên chạy từ xa."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--kernel\', required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--seconds\', type=int, default=35)\n    args = parser.parse_args()\n    owner, slug = args.kernel.split(\'/\')\n    api = KaggleApi()\n    api.authenticate()\n    request = ApiGetKernelSessionLogsStreamRequest()\n    request.user_name = owner\n    request.kernel_slug = slug\n    request.wait_for_logs_url_seconds = 5\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with args.output.open(\'x\') as saved, api.build_kaggle_client() as client:\n        response = client.kernels.kernels_api_client.get_kernel_session_logs_stream(request)\n        print(\'Loại log:\', response.headers.get(\'Content-Type\'), flush=True)\n        messages = []\n        signal.signal(signal.SIGALRM, signal.default_int_handler)\n        signal.alarm(args.seconds)\n        try:\n            for raw in response.iter_lines(chunk_size=1):\n                line = raw.decode(\'utf-8\', \'replace\')\n                saved.write(line + \'\\n\')\n                saved.flush()\n                if line.startswith(\'data: \'):\n                    try:\n                        message = json.loads(line[6:]).get(\'data\', \'\')\n                    except (ValueError, AttributeError):\n                        continue\n                    if any(word in message for word in [\'Đang thử\', \'Đang chạy đủ\', \'Đã chạy\', \'Error\', \'Traceback\']):\n                        messages.append(message.strip())\n        except KeyboardInterrupt:\n            pass\n        finally:\n            signal.alarm(0)\n            response.close()\n        print(\'\\n\'.join(messages[-12:]))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/kaggle_parallel_cli.py': '"""Chạy CLI Kaggle hiện có, upload các phần nhỏ bằng bốn kết nối độc lập."""\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nfrom kaggle.api.kaggle_api_extended import KaggleApi, ResumableUploadContext\nfrom kaggle.cli import main\n\n\nclass RetainedUploadContext(ResumableUploadContext):\n    """Giữ metadata upload để không xóa dữ liệu tạm khi context kết thúc."""\n\n    def __exit__(self, exc_type, exc_value, exc_traceback):\n        """Kết thúc context mà không xóa metadata của lượt upload.\n\n        Args:\n            exc_type: Loại ngoại lệ, nếu có.\n            exc_value: Ngoại lệ, nếu có.\n            exc_traceback: Traceback, nếu có.\n\n        Returns:\n            False để truyền ngoại lệ ra ngoài context.\n        """\n        return False\n\n\ndef upload_one(api, folder, name, blob_type, resources):\n    """Upload một phần qua API của chính package CLI đang cài.\n\n    Args:\n        api: Client đã xác thực của CLI.\n        folder: Thư mục staging.\n        name: Tên tệp dữ liệu.\n        blob_type: Loại blob do CLI chọn.\n        resources: Metadata tài nguyên của CLI.\n\n    Returns:\n        Đối tượng file của request tạo dataset.\n\n    Raises:\n        RuntimeError: Khi một phần không upload thành công.\n    """\n    # Không xóa metadata upload tạm khi kết thúc context.\n    with RetainedUploadContext() as context:\n        uploaded = api._upload_file(name, str(Path(folder) / name), blob_type, context, True, resources)\n    if uploaded is None:\n        raise RuntimeError(f\'Upload thất bại: {name}\')\n    return api._new_file(uploaded)\n\n\ndef parallel_upload_files(api, request, resources, folder, blob_type, upload_context,\n                          quiet=False, dir_mode=\'skip\', ignore_patterns=None):\n    """Upload staging các phần nhỏ, chỉ tạo dataset khi tất cả thành công.\n\n    Args:\n        api: Client CLI đã xác thực.\n        request: Request tạo dataset của CLI.\n        resources: Metadata tệp.\n        folder: Thư mục chứa manifest, các phần và metadata dataset.\n        blob_type: Loại blob do CLI chọn.\n        upload_context: Context của CLI, không dùng để xóa tệp.\n        quiet: Tham số giữ nguyên giao diện CLI.\n        dir_mode: Tham số giữ nguyên giao diện CLI.\n        ignore_patterns: Tham số giữ nguyên giao diện CLI.\n\n    Raises:\n        ValueError: Khi staging chứa tệp ngoài manifest và các phần dữ liệu.\n    """\n    paths = sorted(Path(folder).iterdir())\n    files = [p for p in paths if p.name != \'dataset-metadata.json\']\n    if any(not p.is_file() or not (p.name == \'data_parts.json\' or p.name.endswith(\'.bin\')) for p in files):\n        raise ValueError(\'Staging chỉ được chứa manifest và các phần .bin.\')\n    with ThreadPoolExecutor(max_workers=4) as executor:\n        pending = {executor.submit(upload_one, api, folder, p.name, blob_type, resources): p.name for p in files}\n        for index, future in enumerate(as_completed(pending), 1):\n            request.files.append(future.result())\n            print(f\'Đã upload {index}/{len(files)}: {pending[future]}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    KaggleApi.upload_files = parallel_upload_files\n    main()\n', 'scripts/package_submission.py': '"""Kiểm tra bằng chứng full-frame rồi đóng gói năm kết quả và báo cáo."""\n\nimport argparse\nimport json\nimport subprocess\nimport zipfile\nfrom pathlib import Path\n\nfrom submission_checks import parse_metric_summary, validate_mot_rows\n\n\ndef main() -> None:\n    """Tạo ZIP bài nộp mới, không ghi đè các gói hoặc kết quả hiện có."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run-root\', type=Path, required=True)\n    parser.add_argument(\'--report\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    root = args.run_root\n    counts = json.loads((root / \'data_inventory.json\').read_text())\n    choices = json.loads((root / \'selections.json\').read_text())\n    report = args.report.read_text()\n    if any(marker in report for marker in [\'……………………\', \'(dán output ở đây)\', \'TODO\']):\n        raise ValueError(\'Báo cáo còn mục chưa điền.\')\n    metrics = parse_metric_summary((root / \'metrics_video_1_summary.txt\').read_text())\n    files = {}\n    evidence = {\'metrics_video_1\': metrics, \'videos\': {}}\n    for index in range(1, 6):\n        video = f\'video_{index}\'\n        result = root / \'runs/nop_bai\' / f\'{video}.txt\'\n        rows = validate_mot_rows(result.read_text(), counts[video])\n        log = (result.parent / f\'{video}.log\').read_text()\n        if f\'-> {counts[video]} frame \' not in log:\n            raise ValueError(f\'{video}: thiếu bằng chứng xử lý đủ frame.\')\n        preview = result.parent / f\'{video}_preview.mp4\'\n        count = subprocess.check_output([\'ffprobe\', \'-v\', \'error\', \'-count_frames\',\n            \'-select_streams\', \'v:0\', \'-show_entries\', \'stream=nb_read_frames\',\n            \'-of\', \'default=noprint_wrappers=1:nokey=1\', str(preview)], text=True).strip()\n        if int(count) != counts[video]:\n            raise ValueError(f\'{video}: số frame preview không khớp ảnh gốc.\')\n        files[f\'{video}.txt\'] = result\n        evidence[\'videos\'][video] = {\'processed_frames\': counts[video], \'mot_rows\': rows,\n            \'preview_frames\': int(count), \'configuration\': choices[video]}\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(args.output, \'x\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for name, source in files.items():\n            archive.write(source, name)\n        archive.write(args.report, \'BAO_CAO.md\')\n        archive.writestr(\'KIEM_TRA.json\', json.dumps(evidence, ensure_ascii=False, indent=2))\n        archive.write(root / \'metrics_video_1.log\', \'metrics_video_1.log\')\n        archive.write(root / \'versions.txt\', \'versions.txt\')\n        for name in [\'selections.json\', \'data_inventory.json\', \'runtime_info.json\',\n                     \'weights_sha256.json\', \'trackeval_commit.txt\']:\n            archive.write(root / name, name)\n    print(json.dumps({\'status\': \'SẴN SÀNG NỘP\', \'archive\': str(args.output),\n        \'evidence\': evidence}, ensure_ascii=False, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/prepare_kaggle_dataset.py': '"""Chuẩn bị thư mục upload riêng, giữ nguyên ZIP gốc và kiểm tra gói lab."""\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport zipfile\nfrom pathlib import Path\n\nfrom submission_checks import validate_zip_members\nfrom archive_parts import split_archive\n\n\ndef main() -> None:\n    """Kiểm tra ZIP và tạo staging mới chỉ gồm gói dữ liệu và metadata."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--archive\', required=True, type=Path)\n    parser.add_argument(\'--owner\', required=True)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--part-size-mb\', type=int, default=0)\n    args = parser.parse_args()\n    counts = {}\n    with zipfile.ZipFile(args.archive) as archive:\n        names = archive.namelist()\n        validate_zip_members(names)\n        for index in range(1, 6):\n            video = f\'video_{index}\'\n            images = [name for name in names if f\'{video}/img1/\' in name\n                      and name.endswith(\'.jpg\') and Path(name).stem.isdecimal()\n                      and not name.startswith(\'__MACOSX/\')]\n            if not images:\n                raise ValueError(f\'Thiếu ảnh {video}\')\n            counts[video] = len(images)\n            if index > 1 and any(name.endswith(f\'{video}/gt/gt.txt\') for name in names):\n                raise ValueError(\'Gói học viên không được có nhãn video_2–video_5.\')\n        for required in [\'video_1/gt/gt.txt\', \'video_1/seqinfo.ini\']:\n            if not any(name.endswith(required) for name in names):\n                raise ValueError(f\'Thiếu {required}\')\n        if not any(name.endswith(\'video_1/eval_config.json\') for name in names):\n            print(\'Gói thiếu eval_config.json; script chấm sẽ dùng benchmark LAB, split train.\')\n        corrupt = archive.testzip()\n        if corrupt:\n            raise ValueError(f\'ZIP lỗi CRC: {corrupt}\')\n    args.out.mkdir(parents=True, exist_ok=False)\n    if args.part_size_mb:\n        manifest = split_archive(args.archive, args.out, args.part_size_mb * 1024 * 1024)\n        print(f"Đã chia thành {len(manifest[\'parts\'])} phần, giữ nguyên ZIP gốc.")\n    else:\n        shutil.copy2(args.archive, args.out / \'data_lab21.zip\')\n    metadata = {\'title\': \'Track4 Day22 Tracking Lab Data\', \'id\': args.owner + \'/track4-day22-tracking-lab-data\',\n        \'licenses\': [{\'name\': \'unknown\'}],\n        \'description\': \'Gói dữ liệu giảng viên phát cho bài lab tracking. Dùng riêng tư cho học tập; không xác định giấy phép phân phối lại.\'}\n    (args.out / \'dataset-metadata.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))\n    digest = hashlib.file_digest(args.archive.open(\'rb\'), \'sha256\').hexdigest()\n    print(json.dumps({\'frames\': counts, \'bytes\': args.archive.stat().st_size, \'sha256\': digest,\n        \'upload_directory\': str(args.out)}, ensure_ascii=False, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/run_tracking.py': '#!/usr/bin/env python\n"""Chạy detector YOLO cố định + một tracker (boxmot) trên một video / chuỗi ảnh.\n\nMọi nhóm trong lớp dùng CHUNG detector (``yolo26n.pt``), CHUNG kích thước ảnh đầu\nvào (``IMG_SIZE``) và CHUNG mô hình Re-ID (``REID_WEIGHTS``) cho các tracker có\ndùng ngoại hình. Biến được phép thay đổi là: ``--tracker``, ``--conf`` và ``--iou``\n(ngưỡng của detector, không phải ngưỡng bên trong tracker).\n\nVí dụ:\n    python scripts/run_tracking.py \\\\\n        --source "$LAB_DATA/video_1/img1" \\\\\n        --seq-name video_1 \\\\\n        --tracker bytetrack --conf 0.3 --iou 0.5 \\\\\n        --out runs/nop_bai --save-video --max-frames 150\n\nĐịnh dạng code theo Google Python Style Guide:\nhttps://google.github.io/styleguide/pyguide.html\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport time\nfrom pathlib import Path\nfrom typing import Iterator, Tuple\n\nimport cv2\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\n\nfrom boxmot.tracker_zoo import create_tracker, get_tracker_config\n\n# ---------------------------------------------------------------------------\n# Các biến CỐ ĐỊNH cho cả lớp — KHÔNG sửa khi làm bài chính. Nếu muốn thử\n# nghiệm thêm sau khi đã nộp, hãy ghi rõ trong báo cáo rằng đó là phần mở rộng.\n# ---------------------------------------------------------------------------\nDETECTOR_WEIGHTS = "yolo26n.pt"\nIMG_SIZE = 640\nPERSON_CLASS_ID = 0  # lớp "person" trong COCO — lab chỉ chấm người đi bộ\nREID_WEIGHTS = Path("osnet_x0_25_msmt17.pt")  # tự tải về lần chạy đầu tiên\n\nTRACKER_CHOICES = ["bytetrack", "ocsort", "botsort", "strongsort", "deepocsort"]\nUSES_APPEARANCE = {"botsort", "strongsort", "deepocsort"}\n\n\ndef iter_frames(source: Path) -> Iterator[Tuple[int, np.ndarray]]:\n    """Duyệt frame của thư mục ảnh ``img1/`` hoặc một file video.\n\n    Args:\n        source: Thư mục chứa ảnh ``.jpg`` đặt tên tăng dần, hoặc file ``.mp4``.\n\n    Yields:\n        Cặp ``(frame_index, frame_bgr)``. ``frame_index`` bắt đầu từ 0.\n\n    Raises:\n        FileNotFoundError: Khi thư mục không có ảnh ``.jpg``, hoặc không mở được file video.\n        ValueError: Khi một ảnh không đọc được, tránh bỏ frame âm thầm.\n    """\n    if source.is_dir():\n        frame_paths = sorted(source.glob("*.jpg"))\n        if not frame_paths:\n            raise FileNotFoundError(f"Không tìm thấy ảnh .jpg trong {source}")\n        for i, path in enumerate(frame_paths):\n            frame = cv2.imread(str(path))\n            if frame is None:\n                raise ValueError(f"Không đọc được ảnh {path}")\n            yield i, frame\n    else:\n        cap = cv2.VideoCapture(str(source))\n        if not cap.isOpened():\n            raise FileNotFoundError(f"Không mở được video {source}")\n        i = 0\n        while True:\n            ok, frame = cap.read()\n            if not ok:\n                break\n            yield i, frame\n            i += 1\n        cap.release()\n\n\ndef color_for_id(track_id: int) -> Tuple[int, int, int]:\n    """Sinh một màu BGR ổn định cho mỗi track ID.\n\n    Args:\n        track_id: Định danh track. Cùng ID luôn ra cùng màu.\n\n    Returns:\n        Bộ ba ``(B, G, R)``, mỗi kênh trong khoảng 64–254.\n    """\n    rng = np.random.default_rng(track_id * 9973 + 17)\n    return tuple(int(c) for c in rng.integers(64, 255, size=3))\n\n\ndef detect(detector: YOLO, frame: np.ndarray, conf: float, iou: float) -> np.ndarray:\n    """Chạy detector và trả về hộp người trên một frame.\n\n    Args:\n        detector: Mô hình YOLO đã nạp.\n        frame: Ảnh BGR.\n        conf: Ngưỡng confidence của detector.\n        iou: Ngưỡng IoU cho NMS của detector.\n\n    Returns:\n        Mảng ``(N, 6)`` với mỗi hàng là ``[x1, y1, x2, y2, conf, cls]``.\n        Mảng rỗng ``(0, 6)`` khi không có hộp.\n    """\n    results = detector.predict(\n        frame,\n        conf=conf,\n        iou=iou,\n        imgsz=IMG_SIZE,\n        classes=[PERSON_CLASS_ID],\n        device=next(detector.model.parameters()).device,\n        verbose=False,\n    )[0]\n    if results.boxes is None or len(results.boxes) == 0:\n        return np.empty((0, 6))\n    xyxy = results.boxes.xyxy.cpu().numpy()\n    conf_arr = results.boxes.conf.cpu().numpy()\n    cls_arr = results.boxes.cls.cpu().numpy()\n    return np.hstack([xyxy, conf_arr[:, None], cls_arr[:, None]])\n\n\ndef run(args: argparse.Namespace) -> None:\n    """Detect, track, rồi ghi file kết quả và video xem thử nếu được yêu cầu.\n\n    Args:\n        args: Tham số đã parse, gồm ``source``, ``seq_name``, ``tracker``,\n            ``conf``, ``iou``, ``device``, ``out``, ``save_video``, ``fps``\n            và ``max_frames``.\n    """\n    source = Path(args.source)\n    out_dir = Path(args.out)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    mot_txt = out_dir / f"{args.seq_name}.txt"\n\n    print(f"[nạp mô hình] detector={DETECTOR_WEIGHTS} imgsz={IMG_SIZE} tracker={args.tracker}")\n    if args.tracker in USES_APPEARANCE:\n        print(f"              tracker này dùng Re-ID: {REID_WEIGHTS.name} (tự tải nếu chưa có)")\n    detector = YOLO(DETECTOR_WEIGHTS)\n    detector.to(args.device)\n    tracker = create_tracker(\n        tracker_type=args.tracker,\n        tracker_config=get_tracker_config(args.tracker),\n        reid_weights=REID_WEIGHTS,\n        device=torch.device(args.device),\n        half=False,\n        per_class=False,\n    )\n\n    writer = None\n    rows = []\n    n_frames = 0\n    t0 = time.time()\n\n    for frame_idx, frame in iter_frames(source):\n        if frame is None:\n            continue\n        n_frames += 1\n\n        dets = detect(detector, frame, conf=args.conf, iou=args.iou)\n        tracks = tracker.update(dets, frame)\n\n        for track in tracks:\n            x1, y1, x2, y2, tid = track[0], track[1], track[2], track[3], track[4]\n            tconf = track[5]\n            rows.append(\n                f"{frame_idx + 1},{int(tid)},{x1:.2f},{y1:.2f},{x2 - x1:.2f},"\n                f"{y2 - y1:.2f},{tconf:.4f},-1,-1,-1"\n            )\n\n        if args.save_video:\n            if writer is None:\n                height, width = frame.shape[:2]\n                fourcc = cv2.VideoWriter_fourcc(*"mp4v")\n                preview_path = out_dir / f"{args.seq_name}_preview.mp4"\n                writer = cv2.VideoWriter(str(preview_path), fourcc, args.fps, (width, height))\n            vis = frame.copy()\n            for track in tracks:\n                x1, y1, x2, y2, tid = int(track[0]), int(track[1]), int(track[2]), int(track[3]), int(track[4])\n                color = color_for_id(tid)\n                cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)\n                cv2.putText(\n                    vis, f"ID {tid}", (x1, max(0, y1 - 6)),\n                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2,\n                )\n            writer.write(vis)\n\n        if args.max_frames and n_frames >= args.max_frames:\n            break\n\n    if writer is not None:\n        writer.release()\n\n    mot_txt.write_text("\\n".join(rows) + ("\\n" if rows else ""))\n\n    dt = time.time() - t0\n    fps = n_frames / dt if dt > 0 else 0.0\n    print(\n        f"\\n[{args.seq_name}] tracker={args.tracker} conf={args.conf} iou={args.iou} "\n        f"-> {n_frames} frame trong {dt:.1f}s ({fps:.1f} FPS)"\n    )\n    print(f"  File nộp bài: {mot_txt}")\n    if args.save_video:\n        print(f"  Video xem thử: {out_dir / f\'{args.seq_name}_preview.mp4\'}")\n\n\ndef parse_args() -> argparse.Namespace:\n    """Khai báo tham số dòng lệnh.\n\n    Returns:\n        Namespace chứa các tham số đã parse.\n    """\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--source", required=True, help="Thư mục img1/ hoặc file .mp4")\n    parser.add_argument("--seq-name", required=True, help="Tên video, ví dụ video_1 — dùng làm tên file .txt")\n    parser.add_argument("--tracker", required=True, choices=TRACKER_CHOICES)\n    parser.add_argument("--conf", type=float, default=0.3, help="Ngưỡng confidence của detector")\n    parser.add_argument("--iou", type=float, default=0.5, help="Ngưỡng IoU NMS của detector")\n    parser.add_argument("--device", default="cpu", help="\'cpu\', \'cuda:0\', ...")\n    parser.add_argument("--out", default="runs", help="Thư mục xuất kết quả")\n    parser.add_argument("--save-video", action="store_true", help="Xuất video xem thử")\n    parser.add_argument("--fps", type=int, default=20, help="FPS của video xem thử")\n    parser.add_argument(\n        "--max-frames", type=int, default=0,\n        help="Giới hạn số frame (0 = toàn bộ). Dùng khi thử nhanh; bản nộp phải bỏ tham số này.",\n    )\n    return parser.parse_args()\n\n\nif __name__ == "__main__":\n    run(parse_args())\n', 'scripts/submission_checks.py': '"""Kiểm tra an toàn ZIP và định dạng kết quả MOT, không cần GPU."""\n\nimport math\nfrom pathlib import PurePosixPath\n\n\ndef validate_zip_members(names: list[str]) -> None:\n    """Từ chối đường dẫn ZIP có thể thoát khỏi thư mục giải nén.\n\n    Args:\n        names: Danh sách đường dẫn thành viên ZIP.\n\n    Raises:\n        ValueError: Khi có đường dẫn tuyệt đối, đi ngược hoặc trùng nhau.\n    """\n    if len(set(names)) != len(names):\n        raise ValueError(\'ZIP có đường dẫn trùng nhau.\')\n    for name in names:\n        path = PurePosixPath(name)\n        if path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name or \':\' in name:\n            raise ValueError(f\'Đường dẫn ZIP không hợp lệ: {name}\')\n\n\ndef validate_mot_rows(content: str, frame_count: int) -> int:\n    """Kiểm tra mười cột MOT và tính duy nhất của mỗi ID trong một frame.\n\n    Args:\n        content: Nội dung file kết quả.\n        frame_count: Số frame đã xử lý, không phải số dòng kết quả.\n\n    Returns:\n        Số dòng track hợp lệ.\n\n    Raises:\n        ValueError: Khi file rỗng hoặc có dòng không hợp lệ.\n    """\n    seen = set()\n    lines = content.strip().splitlines()\n    if not lines:\n        raise ValueError(\'File kết quả rỗng, cần kiểm tra trực tiếp.\')\n    for line in lines:\n        fields = line.split(\',\')\n        if len(fields) != 10:\n            raise ValueError(\'Kết quả MOT phải có mười cột.\')\n        values = [float(field) for field in fields]\n        if not all(math.isfinite(value) for value in values):\n            raise ValueError(\'Kết quả có NaN hoặc vô cực.\')\n        frame, track_id = values[:2]\n        if frame != int(frame) or not 1 <= frame <= frame_count or track_id != int(track_id) or track_id < 1:\n            raise ValueError(\'Frame hoặc ID không hợp lệ.\')\n        if values[4] <= 0 or values[5] <= 0 or not 0 <= values[6] <= 1:\n            raise ValueError(\'Kích thước hộp hoặc confidence không hợp lệ.\')\n        key = (int(frame), int(track_id))\n        if key in seen:\n            raise ValueError(\'Một ID xuất hiện hai lần trong cùng frame.\')\n        seen.add(key)\n    return len(lines)\n\n\ndef parse_metric_summary(content: str) -> dict[str, float]:\n    """Đọc ba metric bắt buộc từ bảng summary do TrackEval xuất.\n\n    Args:\n        content: Hai dòng tiêu đề và giá trị của pedestrian_summary.txt.\n\n    Returns:\n        HOTA, MOTA và IDF1 theo thang phần trăm của TrackEval.\n\n    Raises:\n        ValueError: Khi thiếu cột, sai số cột hoặc có số không hợp lệ.\n    """\n    lines = [line for line in content.splitlines() if line.strip()]\n    if len(lines) != 2:\n        raise ValueError(\'Summary TrackEval phải có đúng hai dòng.\')\n    header, values = lines[0].split(), lines[1].split()\n    if len(header) != len(values):\n        raise ValueError(\'Summary lệch số cột.\')\n    fields = dict(zip(header, values))\n    if not {\'HOTA\', \'MOTA\', \'IDF1\'} <= fields.keys():\n        raise ValueError(\'Summary thiếu HOTA, MOTA hoặc IDF1.\')\n    metrics = {key: float(fields[key]) for key in [\'HOTA\', \'MOTA\', \'IDF1\']}\n    if not all(math.isfinite(value) for value in metrics.values()):\n        raise ValueError(\'Metric không hữu hạn.\')\n    if not 0 <= metrics[\'HOTA\'] <= 100 or not 0 <= metrics[\'IDF1\'] <= 100 or metrics[\'MOTA\'] > 100:\n        raise ValueError(\'Metric vượt thang phần trăm hợp lệ.\')\n    return metrics\n', 'tests/test_archive_parts.py': '"""Kiểm tra tính toàn vẹn khi chia và ghép byte, không cần dữ liệu lab."""\n\nimport pytest\n\nfrom archive_parts import assemble_archive, split_archive\n\n\ndef test_roundtrip_keeps_original_bytes(tmp_path):\n    source = tmp_path / \'source.zip\'\n    data = bytes(range(256)) * 3\n    source.write_bytes(data)\n    staging = tmp_path / \'parts\'\n    staging.mkdir()\n    manifest = split_archive(source, staging, 100)\n    assert len(manifest[\'parts\']) == 8\n    result = tmp_path / \'joined.zip\'\n    assemble_archive(staging / \'data_parts.json\', result)\n    assert result.read_bytes() == data\n    assert source.read_bytes() == data\n\n\ndef test_corrupted_part_is_rejected(tmp_path):\n    source = tmp_path / \'source.zip\'\n    source.write_bytes(b\'abcdef\')\n    staging = tmp_path / \'parts\'\n    staging.mkdir()\n    manifest = split_archive(source, staging, 3)\n    (staging / manifest[\'parts\'][0][\'name\']).write_bytes(b\'xyz\')\n    with pytest.raises(ValueError, match=\'checksum\'):\n        assemble_archive(staging / \'data_parts.json\', tmp_path / \'joined.zip\')\n\n\ndef test_join_preserves_existing_destination(tmp_path):\n    source = tmp_path / \'source.zip\'\n    source.write_bytes(b\'abc\')\n    staging = tmp_path / \'parts\'\n    staging.mkdir()\n    split_archive(source, staging, 10)\n    destination = tmp_path / \'joined.zip\'\n    destination.write_bytes(b\'preserve\')\n    with pytest.raises(FileExistsError):\n        assemble_archive(staging / \'data_parts.json\', destination)\n    assert destination.read_bytes() == b\'preserve\'\n', 'tests/test_check_data.py': '"""Tests for check() without lab images or a network."""\n\nfrom pathlib import Path\n\nfrom check_data import check\n\n\ndef _touch_jpg(directory: Path, name: str = "000001.jpg") -> None:\n    directory.mkdir(parents=True, exist_ok=True)\n    (directory / name).write_bytes(b"")\n\n\ndef _lab_tree(root: Path, with_practice_gt: bool = True, extra_images: bool = True) -> None:\n    for index in range(1, 6):\n        name = f"video_{index}"\n        if extra_images or name == "video_1":\n            _touch_jpg(root / name / "img1")\n        else:\n            (root / name).mkdir(parents=True, exist_ok=True)\n    if with_practice_gt:\n        gt_dir = root / "video_1" / "gt"\n        gt_dir.mkdir(parents=True, exist_ok=True)\n        (gt_dir / "gt.txt").write_text("1,1,0,0,10,10,1,1,1\\n", encoding="utf-8")\n\n\ndef test_ready_pack_passes(tmp_path: Path) -> None:\n    _lab_tree(tmp_path)\n    assert check(tmp_path) is True\n\n\ndef test_missing_images_fails(tmp_path: Path) -> None:\n    _lab_tree(tmp_path, extra_images=False)\n    assert check(tmp_path) is False\n\n\ndef test_video_without_labels_still_passes(tmp_path: Path) -> None:\n    _lab_tree(tmp_path)\n    assert not (tmp_path / "video_2" / "gt").exists()\n    assert check(tmp_path) is True\n\n\ndef test_practice_video_without_labels_fails(tmp_path: Path) -> None:\n    _lab_tree(tmp_path, with_practice_gt=False)\n    assert check(tmp_path) is False\n', 'tests/test_evaluate_practice.py': '"""Kiểm tra tiến trình chấm nhận bản vá NumPy và đúng video luyện."""\n\nfrom pathlib import Path\n\nfrom evaluate_practice import _load_eval_config, run_trackeval, stage\n\n\ndef test_subprocess_receives_compatibility_patch(monkeypatch):\n    """Bản vá phải chạy trong tiến trình con, trước khi nạp TrackEval."""\n    calls = []\n    monkeypatch.setattr("evaluate_practice.subprocess.run", lambda cmd, **kw: calls.append((cmd, kw)))\n    run_trackeval(Path("/tmp/trackeval"), "thu_01", "lab", "train")\n    cmd, options = calls[0]\n    assert cmd[1] == "-c"\n    assert "np.float = float; np.int = int" in cmd[2]\n    assert "runpy.run_path" in cmd[2]\n    assert cmd[3].endswith("scripts/run_mot_challenge.py")\n    assert cmd[cmd.index("--SEQ_INFO") + 1] == "video_1"\n    assert options == {"check": True}\n\n\ndef test_stage_respects_configured_split(tmp_path):\n    """Nhãn và dự đoán phải nằm đúng nhánh mà tiến trình chấm đọc."""\n    lab = tmp_path / "lab"\n    video = lab / "video_1"\n    (video / "gt").mkdir(parents=True)\n    (video / "gt/gt.txt").write_text("nhãn thử")\n    (video / "seqinfo.ini").write_text("cấu hình thử")\n    submission = tmp_path / "video_1.txt"\n    submission.write_text("kết quả thử")\n    root = tmp_path / "trackeval"\n    stage(root, lab, submission, "thu_01", "LAB", "validation")\n    assert (root / "data/gt/mot_challenge/LAB-validation/video_1/gt/gt.txt").read_text() == "nhãn thử"\n    assert (root / "data/trackers/mot_challenge/LAB-validation/thu_01/data/video_1.txt").read_text() == "kết quả thử"\n\n\ndef test_missing_eval_config_uses_generic_benchmark(tmp_path):\n    """Gói giảng viên thiếu JSON vẫn chấm được bằng cây thư mục chuẩn."""\n    assert _load_eval_config(tmp_path) == {"benchmark": "LAB", "split": "train"}\n\n\ndef test_existing_eval_config_is_preserved(tmp_path):\n    """Cấu hình có sẵn trong gói luôn được ưu tiên hơn cấu hình dự phòng."""\n    video = tmp_path / "video_1"\n    video.mkdir()\n    (video / "eval_config.json").write_text(\'{"benchmark": "LAB_CUSTOM", "split": "validation"}\')\n    assert _load_eval_config(tmp_path) == {"benchmark": "LAB_CUSTOM", "split": "validation"}\n', 'tests/test_submission_checks.py': '"""Kiểm tra bộ xác minh bài nộp bằng dữ liệu nhỏ, không cần ảnh lab."""\n\nimport pytest\n\nfrom submission_checks import parse_metric_summary, validate_mot_rows, validate_zip_members\n\n\ndef test_safe_archive():\n    validate_zip_members([\'lab_data/video_1/img1/000001.jpg\'])\n\n\n@pytest.mark.parametrize(\'names\', [[\'../outside\'], [\'/outside\'], [\'a\\\\b\'], [\'C:/outside\'], [\'a\', \'a\']])\ndef test_unsafe_archive(names):\n    with pytest.raises(ValueError):\n        validate_zip_members(names)\n\n\ndef test_frames_without_tracks_do_not_require_rows():\n    assert validate_mot_rows(\'1,1,10,20,30,40,0.9,-1,-1,-1\\n\', 100) == 1\n\n\n@pytest.mark.parametrize(\'content\', [\'\', \'1,2\', \'101,1,0,0,10,20,.9,-1,-1,-1\',\n    \'1,1,0,0,-10,20,.9,-1,-1,-1\', \'1,1,nan,0,10,20,.9,-1,-1,-1\',\n    \'1,1,0,0,10,20,.9,-1,-1,-1\\n1,1,0,0,10,20,.9,-1,-1,-1\'])\ndef test_bad_results(content):\n    with pytest.raises(ValueError):\n        validate_mot_rows(content, 100)\n\n\ndef test_metric_summary_keeps_percentage_scale():\n    assert parse_metric_summary(\'HOTA MOTA IDF1 DetA\\n60.5 -10 70.2 55\\n\') == {\n        \'HOTA\': 60.5, \'MOTA\': -10.0, \'IDF1\': 70.2}\n\n\n@pytest.mark.parametrize(\'content\', [\'HOTA MOTA\\n60 70\', \'HOTA MOTA IDF1\\n1 2\',\n    \'HOTA MOTA IDF1\\nnan 3 4\', \'HOTA MOTA IDF1\\n101 3 4\'])\ndef test_invalid_metric_summary(content):\n    with pytest.raises(ValueError):\n        parse_metric_summary(content)\n', 'tests/test_upload_context.py': '"""Kiểm tra context giữ metadata, không cần cài hoặc kết nối Kaggle."""\n\nimport ast\nfrom pathlib import Path\n\nimport pytest\n\n\ndef _context_type():\n    """Nạp riêng lớp context với lớp nền giả không thực hiện xóa tệp."""\n    class FakeContext:\n        cleaned = False\n\n        def __enter__(self):\n            return self\n\n        def __exit__(self, *args):\n            self.cleaned = True\n\n    source = Path(__file__).resolve().parents[1] / \'scripts/kaggle_parallel_cli.py\'\n    tree = ast.parse(source.read_text())\n    definition = next(node for node in tree.body if isinstance(node, ast.ClassDef)\n                      and node.name == \'RetainedUploadContext\')\n    namespace = {\'ResumableUploadContext\': FakeContext}\n    module = ast.Module(body=[definition], type_ignores=[])\n    exec(compile(module, str(source), \'exec\'), namespace)\n    return namespace[\'RetainedUploadContext\']\n\n\ndef test_success_does_not_clean_upload_metadata():\n    context = _context_type()()\n    with context:\n        pass\n    assert context.cleaned is False\n\n\ndef test_upload_error_is_not_suppressed():\n    context = _context_type()()\n    with pytest.raises(RuntimeError, match=\'lỗi upload\'):\n        with context:\n            raise RuntimeError(\'lỗi upload\')\n    assert context.cleaned is False\n', 'pytest.ini': '[pytest]\npythonpath = scripts\n', 'on_tap_metrics.ipynb': '{\n  "cells": [\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "# Ôn metric và xem detector một frame\\n",\n        "\\n",\n        "Chạy notebook này **sau** khi đã `conda activate cv_robotics_lab21` và `export LAB_DATA=...` (xem `HUONG_DAN.md`).\\n",\n        "\\n",\n        "## Ba thước đo không kể cùng một câu chuyện\\n",\n        "\\n",\n        "Chuỗi dưới đây là ID mà tracker gán cho **một** người thật qua 10 frame. Dấu `.` nghĩa là tracker **bỏ sót** frame đó (không có hộp).\\n",\n        "\\n",\n        "- **MOTA** phạt lỗi theo **số lần**: bỏ sót, hộp giả, và mỗi lần đổi ID. Đổi ID một lần rồi giữ sai đến hết chỉ bị trừ một lần. Đổi ID liên tục bị trừ nhiều lần.\\n",\n        "- **IDF1** nhìn cả track dài. Gán sai danh tính rồi giữ sai đến hết làm IDF1 tụt, dù MOTA vẫn cao.\\n",\n        "- **HOTA** cân bằng hai việc: phát hiện đúng người (Detection) và giữ đúng danh tính (Association). Hai tracker cùng kém ở phần giữ ID có thể có HOTA gần nhau dù MOTA khác xa.\\n",\n        "\\n",\n        "Số trong bảng đã tính sẵn trên ví dụ ngắn. Bạn đọc bảng và trả lời, không cần sửa công thức."\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "bang = [\\n",\n        "    # tracker, chuỗi ID, MOTA, IDF1, HOTA, số lần đổi ID\\n",\n        "    (\\"A\\", \\"1111112222\\", 0.90, 0.60, 0.72, 1),\\n",\n        "    (\\"B\\", \\"1212121212\\", 0.10, 0.50, 0.71, 9),\\n",\n        "    (\\"C\\", \\"111111....\\", 0.60, 0.75, 0.60, 0),\\n",\n        "    (\\"D\\", \\"1111111111\\", 1.00, 1.00, 1.00, 0),\\n",\n        "]\\n",\n        "print(f\\"{\'Tracker\':8} {\'Chuỗi\':12} {\'MOTA\':6} {\'IDF1\':6} {\'HOTA\':6} {\'Đổi ID\':6}\\")\\n",\n        "for row in bang:\\n",\n        "    ten, chuoi, mota, idf1, hota, doi_id = row\\n",\n        "    print(f\\"{ten:8} {chuoi:12} {mota:6.2f} {idf1:6.2f} {hota:6.2f} {doi_id:6d}\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "Điền `True` hoặc `False`, rồi chạy ô kiểm tra.\\n",\n        "\\n",\n        "1. MOTA phạt tracker B nặng vì B đổi ID rất nhiều lần, dù mỗi lần đổi chỉ kéo dài một frame.\\n",\n        "2. HOTA phân biệt A và B rõ hơn IDF1.\\n",\n        "3. Tracker C bỏ sót bốn frame cuối nhưng không đổi ID, nên IDF1 của C cao hơn A trong khi MOTA của C thấp hơn A."\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "nhan_dinh_1 = True  # MOTA phạt mạnh số lần đổi ID\\n",\n        "nhan_dinh_2 = False  # HOTA tách A và B rõ hơn IDF1\\n",\n        "nhan_dinh_3 = True  # C: IDF1 cao hơn A, MOTA thấp hơn A\\n",\n        "\\n",\n        "dap_an = {1: True, 2: False, 3: True}\\n",\n        "ban_chon = {1: nhan_dinh_1, 2: nhan_dinh_2, 3: nhan_dinh_3}\\n",\n        "for cau, dung in dap_an.items():\\n",\n        "    if ban_chon[cau] is None:\\n",\n        "        print(f\\"Câu {cau}: chưa điền\\")\\n",\n        "    elif ban_chon[cau] == dung:\\n",\n        "        print(f\\"Câu {cau}: đúng\\")\\n",\n        "    else:\\n",\n        "        print(f\\"Câu {cau}: xem lại bảng — HOTA của A và B gần nhau, IDF1 cách xa hơn\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "## Detector chưa phải tracker\\n",\n        "\\n",\n        "YOLO nhìn **từng ảnh một**. Nó vẽ hộp quanh người và cho một điểm tin cậy. Nó **không** nhớ người ở frame trước, nên **không có ID**.\\n",\n        "\\n",\n        "Tracker nhận các hộp đó rồi mới gán ID theo thời gian. Đổi `--conf` hoặc `--iou` là đổi ngưỡng của YOLO (hộp nào được giữ), không phải ngưỡng bên trong tracker.\\n",\n        "\\n",\n        "Hai ô dưới chạy `yolo26n.pt` trên **một** ảnh của `video_1`. Cả lớp dùng chung trọng số này, kích thước ảnh 640 px, chỉ lớp người."\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "id": "9f259666",\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "import os\\n",\n        "from pathlib import Path\\n",\n        "\\n",\n        "import cv2\\n",\n        "import matplotlib.pyplot as plt\\n",\n        "from ultralytics import YOLO\\n",\n        "\\n",\n        "IMG_SIZE = 640\\n",\n        "PERSON_CLASS_ID = 0\\n",\n        "DETECTOR_WEIGHTS = \\"yolo26n.pt\\"\\n",\n        "\\n",\n        "lab_data = os.environ.get(\\"LAB_DATA\\", \\"\\").strip()\\n",\n        "frame_path = None\\n",\n        "if not lab_data:\\n",\n        "    print(\\"Chưa có biến LAB_DATA.\\")\\n",\n        "    print(\\"Trong terminal, trước khi mở notebook:\\")\\n",\n        "    print(\'  export LAB_DATA=/đường/dẫn/lab_data\')\\n",\n        "    print(\\"Rồi chọn kernel cv_robotics_lab21 và chạy lại từ ô này.\\")\\n",\n        "else:\\n",\n        "    frames = sorted((Path(lab_data) / \\"video_1\\" / \\"img1\\").glob(\\"*.jpg\\"))\\n",\n        "    if not frames:\\n",\n        "        print(f\\"Không thấy ảnh jpg trong {lab_data}/video_1/img1\\")\\n",\n        "    else:\\n",\n        "        frame_path = frames[0]\\n",\n        "        print(f\\"Ảnh: {frame_path.name}\\")\\n",\n        "\\n",\n        "detector = None\\n",\n        "frame_bgr = None\\n",\n        "if frame_path is not None:\\n",\n        "    frame_bgr = cv2.imread(str(frame_path))\\n",\n        "    detector = YOLO(DETECTOR_WEIGHTS)\\n",\n        "    result = detector.predict(\\n",\n        "        frame_bgr,\\n",\n        "        conf=0.3,\\n",\n        "        iou=0.5,\\n",\n        "        imgsz=IMG_SIZE,\\n",\n        "        classes=[PERSON_CLASS_ID],\\n",\n        "        verbose=False,\\n",\n        "    )[0]\\n",\n        "    n_boxes = 0 if result.boxes is None else len(result.boxes)\\n",\n        "    print(f\\"conf=0.3, iou=0.5 -> {n_boxes} hộp người. Chưa có ID.\\")\\n",\n        "    vis = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)\\n",\n        "    plt.figure(figsize=(12, 7))\\n",\n        "    plt.imshow(vis)\\n",\n        "    plt.axis(\\"off\\")\\n",\n        "    plt.title(\\"YOLO26n, conf=0.3 — hộp phát hiện, không có track ID\\")\\n",\n        "    plt.show()\\n"\n      ]\n    },\n    {\n      "cell_type": "code",\n      "execution_count": null,\n      "metadata": {},\n      "outputs": [],\n      "source": [\n        "if detector is None or frame_bgr is None:\\n",\n        "    print(\\"Bỏ qua ô này. Chạy ô detector phía trên khi đã có LAB_DATA.\\")\\n",\n        "else:\\n",\n        "    for conf in (0.15, 0.5):\\n",\n        "        result = detector.predict(\\n",\n        "            frame_bgr,\\n",\n        "            conf=conf,\\n",\n        "            iou=0.5,\\n",\n        "            imgsz=IMG_SIZE,\\n",\n        "            classes=[PERSON_CLASS_ID],\\n",\n        "            verbose=False,\\n",\n        "        )[0]\\n",\n        "        n_boxes = 0 if result.boxes is None else len(result.boxes)\\n",\n        "        print(f\\"conf={conf:.2f} -> {n_boxes} hộp người\\")\\n",\n        "    print(\\"conf thấp thường ra nhiều hộp hơn, kể cả hộp trên nền. conf cao dễ bỏ sót người nhỏ hoặc tối.\\")\\n"\n      ]\n    },\n    {\n      "cell_type": "markdown",\n      "metadata": {},\n      "source": [\n        "## Sang bước tracking\\n",\n        "\\n",\n        "ID chỉ xuất hiện khi bạn chạy `scripts/run_tracking.py`. Script đó lấy hộp YOLO rồi đưa vào tracker (`bytetrack`, `ocsort`, `botsort`, `strongsort`, hoặc `deepocsort`).\\n",\n        "\\n",\n        "Khi viết báo cáo, tách hai loại lỗi:\\n",\n        "\\n",\n        "- Đổi ID liên tục (kiểu B) làm MOTA tụt mạnh.\\n",\n        "- Đổi ID một lần rồi giữ sai (kiểu A) vẫn có MOTA cao. IDF1 và video lộ hậu quả rõ hơn.\\n",\n        "- Bỏ sót người (kiểu C) khác với gán nhầm người.\\n",\n        "\\n",\n        "Chỉ `video_1` có bảng số. Bốn video kia bạn mô tả đúng kiểu lỗi này bằng mắt."\n      ]\n    }\n  ],\n  "metadata": {\n    "kernelspec": {\n      "display_name": "Python 3",\n      "language": "python",\n      "name": "python3"\n    },\n    "language_info": {\n      "name": "python",\n      "pygments_lexer": "ipython3"\n    }\n  },\n  "nbformat": 4,\n  "nbformat_minor": 5\n}\n', 'requirements-kaggle.txt': '# Phiên bản đã chạy thành công trên GPU Kaggle, Python 3.13.\n# Dùng PyTorch/CUDA có sẵn của Kaggle; ghi phiên bản thực tế vào versions.txt.\n# BoxMOT cài riêng: pip install --no-deps boxmot==10.0.42\nfilterpy==1.4.5\nftfy==6.3.1\ngdown==5.2.2\nGitPython==3.2.0\nlapx==0.10.0\nloguru==0.7.3\nmatplotlib==3.10.0\nnumpy==2.1.3\nopencv-python==4.14.0.94\npandas==2.3.3\npytest==8.4.2\nPyYAML==6.0.3\nregex==2025.11.3\nscikit-learn==1.6.1\nscipy==1.16.3\ntensorboard==2.20.0\nultralytics==8.4.174\nyacs==0.1.8\n'}
for name, content in SOURCES.items():
    destination = WORK / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content)
for weight in ['yolo26n.pt', 'osnet_x0_25_msmt17.pt']:
    cached = list(Path('/kaggle/input').rglob(weight))
    if len(cached) == 1 and not (WORK / weight).exists():
        shutil.copy2(cached[0], WORK / weight)
        print(f'Dùng lại trọng số đã kiểm thử: {weight}')
# Kaggle hiện chạy Python 3.13: giữ thư viện GPU có sẵn của phiên.
# BoxMOT cũ pin NumPy 1.23.1 không có wheel tương thích; cài riêng
# không kéo dependency, rồi kiểm tra đủ năm tracker với NumPy của Kaggle.
PYTHON = sys.executable
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-deps', 'boxmot==10.0.42'], check=True)
subprocess.run([PYTHON, '-m', 'pytest', '-q'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'freeze'], check=True, stdout=(WORK / 'versions.txt').open('w'))


In [ ]:
(WORK / 'selections.json').write_text('{"video_1": {"tracker": "botsort", "conf": 0.3, "iou": 0.5}, "video_2": {"tracker": "botsort", "conf": 0.15, "iou": 0.5}, "video_3": {"tracker": "botsort", "conf": 0.15, "iou": 0.5}, "video_4": {"tracker": "botsort", "conf": 0.3, "iou": 0.5}, "video_5": {"tracker": "bytetrack", "conf": 0.3, "iou": 0.5}}')
os.environ['LAB_MODE'] = 'final'
subprocess.run([PYTHON, 'scripts/kaggle_lab_pipeline.py'], check=True)
